# 02. CoinMarketCap 시가총액 + 공급량 + OHLCV 수집

**출처**: CoinMarketCap 내부 웹 API (무료, API 키 불필요)  
**수집 대상**: USDT, USDC, DAI, BTC, ETH  
**기간**: 2019-11-22 ~ 현재  
**저장 위치**: `data/raw/cmc_market_info.csv`

**수집 이유**:  
- 시가총액 변화 = 스테이블코인의 시장 신뢰도 지표  
- 공급량 데이터 = 대규모 발행/소각 여부 파악  
- CoinGecko 무료 API는 최근 365일만 지원 → CoinMarketCap 내부 API 활용

**API 구조**:  
`https://api.coinmarketcap.com/data-api/v3/cryptocurrency/historical`  
- 한 번에 최대 약 180일 분량 반환 → 180일 단위로 청크 분할 수집

In [ ]:
import requests
import pandas as pd
import time
import os
from datetime import datetime, timedelta

SAVE_DIR   = os.path.join('..', 'data', 'raw')

# CoinMarketCap 코인 고유 ID (웹 URL에서 확인 가능)
COINS = {
    'USDT': '825',
    'USDC': '3408',
    'DAI':  '4943',
    'BTC':  '1',
    'ETH':  '1027'
}

START_DATE = datetime(2019, 11, 22)
END_DATE   = datetime.today()
CHUNK_DAYS = 180  # API 한 번 호출당 180일 데이터 수집

# 브라우저처럼 보이도록 헤더 설정 (bot 차단 우회)
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'application/json'
}

In [ ]:
def fetch_chunk(coin_id, start_dt, end_dt):
    """
    CoinMarketCap API 단일 청크 요청
    - convertId=2781: USD 기준으로 환산
    - timeStart/timeEnd: Unix 타임스탬프 (초 단위)
    """
    url = 'https://api.coinmarketcap.com/data-api/v3/cryptocurrency/historical'
    params = {
        'id':        coin_id,
        'convertId': '2781',  # 2781 = USD
        'timeStart': int(start_dt.timestamp()),
        'timeEnd':   int(end_dt.timestamp())
    }
    res = requests.get(url, headers=HEADERS, params=params, timeout=30)
    res.raise_for_status()
    return res.json()


def parse_chunk(data):
    """
    API 응답 JSON → DataFrame 변환
    - quotes 배열에서 날짜별 OHLCV + 시가총액 추출
    - circulating_supply는 응답 최상단에 단일값으로 존재 (일별 아님)
    """
    rows = []
    for q in data.get('data', {}).get('quotes', []):
        quote = q.get('quote', {})
        rows.append({
            'Date':               q['timeOpen'][:10],  # YYYY-MM-DD 형식으로 자름
            'open':               quote.get('open'),
            'high':               quote.get('high'),
            'low':                quote.get('low'),
            'close':              quote.get('close'),
            'volume':             quote.get('volume'),
            'market_cap':         quote.get('marketCap'),
            'circulating_supply': data.get('data', {}).get('circulatingSupply'),
            'total_supply':       data.get('data', {}).get('totalSupply'),
            'max_supply':         data.get('data', {}).get('maxSupply'),
        })
    return pd.DataFrame(rows)


def fetch_all(coin_id, symbol):
    """
    전체 기간을 180일 단위 청크로 나눠 수집 후 합치기
    - API 과부하 방지를 위해 청크 간 1.5초 대기
    """
    all_rows = []
    current  = START_DATE

    while current < END_DATE:
        chunk_end = min(current + timedelta(days=CHUNK_DAYS), END_DATE)
        try:
            data  = fetch_chunk(coin_id, current, chunk_end)
            chunk = parse_chunk(data)
            all_rows.append(chunk)
            print(f'  {current.strftime("%Y-%m-%d")} ~ {chunk_end.strftime("%Y-%m-%d")}: {len(chunk)}개')
        except Exception as e:
            print(f'  오류 ({current.strftime("%Y-%m-%d")}): {e}')
        current = chunk_end + timedelta(days=1)
        time.sleep(1.5)  # 서버 부하 방지

    if not all_rows:
        return pd.DataFrame()

    df = pd.concat(all_rows, ignore_index=True)
    # 날짜 중복 제거 후 오름차순 정렬
    df = df.drop_duplicates(subset='Date').sort_values('Date').reset_index(drop=True)

    # 모든 컬럼명에 코인 심볼 추가 (예: open → open_USDT)
    df = df.rename(columns={
        'open':               f'open_{symbol}',
        'high':               f'high_{symbol}',
        'low':                f'low_{symbol}',
        'close':              f'close_{symbol}',
        'volume':             f'volume_{symbol}',
        'market_cap':         f'market_cap_{symbol}',
        'circulating_supply': f'circulating_supply_{symbol}',
        'total_supply':       f'total_supply_{symbol}',
        'max_supply':         f'max_supply_{symbol}',
    })
    return df

In [ ]:
# 전체 수집 실행
# ⚠️ 수집 시간 약 5~10분 소요 (코인 5개 × 180일 청크)
os.makedirs(SAVE_DIR, exist_ok=True)
all_dfs = []

for symbol, coin_id in COINS.items():
    print(f'\n수집 중: {symbol} (id={coin_id})')
    df = fetch_all(coin_id, symbol)
    if df.empty:
        print(f'  {symbol} 데이터 없음')
        continue
    all_dfs.append(df)
    print(f'  총 {len(df)}개 행 ({df["Date"].min()} ~ {df["Date"].max()})')
    time.sleep(2)  # 코인 간 추가 대기

In [ ]:
# 5개 코인 데이터를 Date 기준으로 가로 병합 (wide format)
# outer join: 어느 코인이라도 해당 날짜 데이터가 있으면 포함
merged = all_dfs[0]
for df in all_dfs[1:]:
    merged = merged.merge(df, on='Date', how='outer')
merged = merged.sort_values('Date').reset_index(drop=True)

save_path = os.path.join(SAVE_DIR, 'cmc_market_info.csv')
merged.to_csv(save_path, index=False)
print(f'저장 완료: {save_path}')
print(f'총 {len(merged)}개 행, {len(merged.columns)}개 컬럼')
merged.head(3)